# GRPO Source Code

## GRPO

![](../images/grpo1.png)

For each question $q$, GRPO samples a
group of outputs $\{o_1, o_2, \dots , o_G\}$ from the old policy $\pi_{\theta_{old}}$ and then optimizes the policy model
by maximizing the following objective:

$$
\begin{aligned}
J_{GRPO}(\theta) = &\underset{q\sim P(Q),o\sim\pi_{\theta_{\text{old}}}(O|q)}{\mathbb{E}}\frac{1}{G}\sum_{i=1}^{G}\frac{1}{|o_{i}|}\sum_{t=1}^{|o_{i}|}\{\min\\
&\left[\frac{\pi_{\theta}(o_{i,t}|q,o_{i}<t)}{\pi_{\theta_{\text{old}}}(o_{i,t}|q,o_{i}<t)}\hat{A}_{i,t}, \text{clip}\left(\frac{\pi_{\theta}(o_{i,t}|q,o_{i}<t)}{\pi_{\theta_{\text{old}}}(o_{i,t}|q,o_{i}<t)}, 1-\epsilon, 1+\epsilon\right)\hat{A}_{i,t}\right] - \beta\mathbb{D}_{KL}[\pi_{\theta}||\pi_{ref}]\},
\end{aligned}
$$

where $\epsilon$ and $\beta$ are hyper-parameters, and <span style="color: red">$\hat{A}_{i,t}$ is the advantage calculated based on relative
rewards of the outputs inside each group only.</span> Also note that, instead of adding KL
penalty in the reward, GRPO regularizes by directly adding the KL divergence between the
trained policy and the reference policy to the loss, avoiding complicating the calculation of $\hat{A}_{i,t}$. And different from the KL penalty term used in PPO, we estimate the KL divergence with the
following unbiased estimator:

$$\mathbb{D}_{KL}[\pi_{\theta}||\pi_{ref}] = \frac{\pi_{\text{ref}}(o_{i,t}|q, o_{i}<t)}{\pi_{\theta}(o_{i,t}|q,o_{i}<t)} - \log\frac{\pi_{\text{ref}}(o_{i,t}|q, o_{i}<t)}{\pi_{\theta}(o_{i,t}|q,o_{i}<t)} - 1,$$

## Config

### Data

* `data.train_files`: Training set parquet. Can be a list or a single file. The program will read all files into memory, so it can’t be too large (< 100GB).
* `data.val_files`: Validation parquet.
* `data.train_batch_size`: Batch size sampled for one training iteration of different RL algorithms.
* `data.prompt_key`: The field in the dataset where the prompt is located. Default is ‘prompt’.
* `data.max_prompt_length`: Maximum prompt length. All prompts will be <span style="color: red">left-padded</span> to this length.
* `data.filter_overlong_prompts`: Default don’t filter.

### Actor/Rollout/Reference Policy

* `actor_rollout_ref.model.path`: Huggingface model path. This can be either local path or HDFS path.
* `actor_rollout_ref.actor.strategy`: fsdp or megatron.
* `custom_reward_function.path`: The path to the file containing your customized reward function. If not specified, pre-implemented reward functions will be used.
* `custom_reward_function.name` (Optional) : The name of the reward function within the specified file. Default is ‘compute_score’.
* `trainer.total_epochs`: Number of epochs in training.

## Fit

```python
for epoch in range(self.config.trainer.total_epochs):
    for batch_dict in self.train_dataloader:
        # rollout
        gen_batch_output = self.actor_rollout_wg.generate_sequences(gen_batch)
        batch = batch.repeat(repeat_times=self.config.actor_rollout_ref.rollout.n, interleave=True)
        
```